In [2]:
import os
import random
import shutil
from pathlib import Path
from functools import partial
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

import cv2
import numpy as np
import albumentations as A

# ----------------- CONFIG -----------------
SRC_DIR = Path("./data/train")                 # โฟลเดอร์ต้นทาง: ./data/train/<class>/*
OUT_DIR = Path("./data/train_augmented")       # โฟลเดอร์ปลายทาง
IMAGE_SIZE = (224, 224)                        # ขนาดที่ต้องการ (H, W)
TARGET_COUNT_PER_CLASS = None                  # ถ้า None => ใช้ multiplier
MULTIPLIER = 3                                 # แต่ละคลาสจะมีรูปเท่าเดิม * multiplier (ถ้า TARGET_COUNT_PER_CLASS None)
AUG_PER_IMAGE = 3                              # ถ้าอยากคงจำนวนเดิมแต่สร้างต่อภาพละกี่รูป (ใช้ปรับแต่ง)
SEED = 42
NUM_WORKERS = 8                                # ปรับตามเครื่องคุณ
VERBOSE = True

# สร้าง output dir
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ตั้ง seed เพื่อ reproducibility
random.seed(SEED)
np.random.seed(SEED)

# ----------------- AUGMENTATION PIPELINE -----------------
# ปรับ pipeline ตามความสมจริงของ dataset ของคุณ
transform = A.Compose([
    A.RandomResizedCrop(
    size=IMAGE_SIZE,   # (height, width)
    scale=(0.8, 1.0),
    p=0.6
    ),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.08, scale_limit=0.12, rotate_limit=25, border_mode=cv2.BORDER_REFLECT_101, p=0.7),
    A.OneOf([
        A.GaussNoise(var_limit=(10.0, 50.0)),
        A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5))
    ], p=0.3),
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.7),
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8,8), p=0.5)
    ], p=0.6),
    A.OneOf([
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=10),
        A.RGBShift(r_shift_limit=10, g_shift_limit=10, b_shift_limit=10)
    ], p=0.4),
    A.CoarseDropout(max_holes=1, max_height=int(0.08*IMAGE_SIZE[0]), max_width=int(0.08*IMAGE_SIZE[1]), p=0.2),
], p=1.0)

# ----------------- HELPERS -----------------
def list_image_files(folder):
    exts = {".jpg", ".jpeg", ".png", ".bmp"}
    return [p for p in Path(folder).iterdir() if p.suffix.lower() in exts and p.is_file()]

def ensure_class_folder(class_name):
    p = OUT_DIR / class_name
    p.mkdir(parents=True, exist_ok=True)
    return p

def read_and_resize(path, size):
    img = cv2.imread(str(path))
    if img is None:
        raise ValueError(f"Bad image: {path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (size[1], size[0]), interpolation=cv2.INTER_AREA)
    return img

def save_image_rgb(path, img_rgb):
    # convert RGB -> BGR for OpenCV write
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    cv2.imwrite(str(path), img_bgr)

def augment_and_save_single(src_path, dest_folder, aug_index, orig_basename):
    try:
        img = read_and_resize(src_path, IMAGE_SIZE)
        augmented = transform(image=img)['image']
        out_name = f"{orig_basename}_aug{aug_index}_{random.randint(0,99999)}.jpg"
        out_path = dest_folder / out_name
        save_image_rgb(out_path, augmented)
        return True
    except Exception as e:
        if VERBOSE:
            print(f"[WARN] {src_path} -> failed: {e}")
        return False

# ----------------- MAIN LOGIC -----------------
def main():
    classes = [d for d in SRC_DIR.iterdir() if d.is_dir()]
    if not classes:
        print("No class folders found in", SRC_DIR)
        return

    # สร้างโฟลเดอร์ปลายทางและคัดลอกไฟล์เดิม (option: หรือจะไม่คัดลอก ถ้าต้องการแยก)
    for c in classes:
        dest = ensure_class_folder(c.name)
        # คัดลอกต้นฉบับไปยัง out (comment ถ้าต้องการเก็บแยก)
        for img_path in list_image_files(c):
            shutil.copy(img_path, dest / img_path.name)

    # คำนวณ target per class
    class_counts = {c.name: len(list_image_files(c)) for c in classes}
    if TARGET_COUNT_PER_CLASS is None:
        target_counts = {name: max(1, class_counts[name] * MULTIPLIER) for name in class_counts}
    else:
        target_counts = {name: TARGET_COUNT_PER_CLASS for name in class_counts}

    if VERBOSE:
        print("Initial counts:", class_counts)
        print("Target counts:", target_counts)

    # ทำ augmentation ตามความต้องการ
    for c in classes:
        name = c.name
        dest = OUT_DIR / name
        current = len(list_image_files(dest))
        target = target_counts[name]
        if current >= target:
            if VERBOSE:
                print(f"[SKIP] {name} has {current} >= {target}")
            continue

        needed = target - current
        src_files = list_image_files(c)
        if not src_files:
            print(f"[ERROR] no source images for class {name}, skipping")
            continue

        # We'll loop through source images, augmenting each up to AUG_PER_IMAGE as needed
        pbar = tqdm(total=needed, desc=f"Augment {name}", unit="img")
        aug_count = 0
        idx = 0

        # Use ThreadPool for IO-bound saving
        with ThreadPoolExecutor(max_workers=NUM_WORKERS) as ex:
            futures = []
            while aug_count < needed:
                src_path = src_files[idx % len(src_files)]
                orig_basename = src_path.stem
                # determine how many to create from this source in this pass
                to_create = min(AUG_PER_IMAGE, needed - aug_count)
                for k in range(to_create):
                    # schedule augmentation
                    futures.append(ex.submit(augment_and_save_single, src_path, dest, k, orig_basename))
                    aug_count += 1
                    pbar.update(1)
                idx += 1

            # ensure all done
            for fut in tqdm(futures, desc=f"Saving {name}", leave=False):
                _ = fut.result()
            pbar.close()

        if VERBOSE:
            print(f"[DONE] {name}: created {needed} images -> now {len(list_image_files(dest))}")

    print("All augmentation finished. Output folder:", OUT_DIR)

if __name__ == "__main__":
    main()


C:\Users\ahmad\AppData\Roaming\Python\Python311\site-packages\albumentations\core\validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
C:\Users\ahmad\AppData\Local\Temp\ipykernel_2228\3548450575.py:42: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0)),
C:\Users\ahmad\AppData\Local\Temp\ipykernel_2228\3548450575.py:53: UserWarning: Argument(s) 'max_holes, max_height, max_width' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=1, max_height=int(0.08*IMAGE_SIZE[0]), max_width=int(0.08*IMAGE_SIZE[1]), p=0.2),


Initial counts: {'ALGAL_LEAF_SPOT': 632, 'ALLOCARIDARA_ATTACK': 787, 'HEALTHY_LEAF': 841, 'LEAF_BLIGHT': 808, 'PHOMOPSIS_LEAF_SPOT': 757}
Target counts: {'ALGAL_LEAF_SPOT': 1896, 'ALLOCARIDARA_ATTACK': 2361, 'HEALTHY_LEAF': 2523, 'LEAF_BLIGHT': 2424, 'PHOMOPSIS_LEAF_SPOT': 2271}


Augment ALGAL_LEAF_SPOT: 100%|██████████| 1264/1264 [00:01<00:00, 742.15img/s]


[DONE] ALGAL_LEAF_SPOT: created 1264 images -> now 1896


Augment ALLOCARIDARA_ATTACK: 100%|██████████| 1574/1574 [00:02<00:00, 715.12img/s]


...

[DONE] HEALTHY_LEAF: created 1682 images -> now 2523


Augment LEAF_BLIGHT: 100%|██████████| 1616/1616 [00:02<00:00, 737.12img/s]


[DONE] LEAF_BLIGHT: created 1616 images -> now 2424


Augment PHOMOPSIS_LEAF_SPOT: 100%|██████████| 1514/1514 [00:02<00:00, 711.52img/s]


[DONE] PHOMOPSIS_LEAF_SPOT: created 1514 images -> now 2271
All augmentation finished. Output folder: data\train_augmented
